# ModelGuard – AI Lifecycle Management & Governance Agent

## Notebook 03: Model Monitoring and Drift Detection

## Notebook Overview

This notebook focuses on monitoring the trained customer churn prediction model. The main objective is to identify changes in incoming data that may affect the reliability of the model over time.

The monitoring process compares new data with the reference data used during model development. It also evaluates model performance when actual customer outcomes become available.

## Main Objectives

- Establish reference data for monitoring.
- Prepare current data for comparison.
- Detect changes in numerical and categorical feature distributions.
- Monitor model performance using actual outcomes.
- Generate monitoring reports and identify potential risks.

### Environment Setup

Imports the required libraries and defines the main project directories. These settings will be used throughout the monitoring process to access datasets, trained models, and monitoring outputs.

In [2]:
# Import required libraries

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Define project directories

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"
MONITORING_DIR = PROJECT_ROOT / "monitoring"

# Create monitoring directory if it does not exist

MONITORING_DIR.mkdir(parents=True, exist_ok=True)

# Display project paths

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Artifacts Directory:", ARTIFACTS_DIR)
print("Models Directory:", MODELS_DIR)
print("Monitoring Directory:", MONITORING_DIR)

print("\nEnvironment setup completed successfully.")

Project Root: C:\Users\User\ModelGuard
Data Directory: C:\Users\User\ModelGuard\data
Artifacts Directory: C:\Users\User\ModelGuard\artifacts
Models Directory: C:\Users\User\ModelGuard\models
Monitoring Directory: C:\Users\User\ModelGuard\monitoring

Environment setup completed successfully.


## Inspect Available Data Files

Examines the available project data files. The purpose is to identify the datasets that can be used as reference data for monitoring and to verify their availability before starting the drift detection process.

In [3]:
# List available files in the project data directory

print("Available files in the data directory:\n")

for file_path in sorted(DATA_DIR.rglob("*")):
    if file_path.is_file():
        print(
            f"File: {file_path.relative_to(PROJECT_ROOT)} "
            f"| Size: {file_path.stat().st_size / 1024:.2f} KB"
        )

Available files in the data directory:

File: data\raw\.ipynb_checkpoints\Telco-Customer-Churn-checkpoint.csv | Size: 947.71 KB
File: data\raw\Telco-Customer-Churn.csv | Size: 947.71 KB


### Prepare Reference Data

The original Telco Customer Churn dataset will be used to establish the reference data for monitoring.

To maintain consistency with the previous model development process, the same data cleaning procedure and train-test split will be applied. The training features will serve as the reference data, while the test features will be kept separate for later monitoring experiments.

#### Load and Prepare the Dataset

In [4]:
from sklearn.model_selection import train_test_split

# Load the original dataset
df_raw = pd.read_csv(DATA_DIR / "raw" / "Telco-Customer-Churn.csv")

# Create a working copy
df = df_raw.copy()

# Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype("string").str.strip(),
    errors="coerce"
)

# Replace missing TotalCharges with zero
# These records have zero tenure
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Separate features and target
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

# Reproduce the original train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset preparation completed.")

print("\nReference Data Shape:", X_train.shape)
print("Test Data Shape:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nMissing Values in Reference Data:", X_train.isnull().sum().sum())

Dataset preparation completed.

Reference Data Shape: (5634, 19)
Test Data Shape: (1409, 19)

Training Target Distribution:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64

Missing Values in Reference Data: 0


### Save Reference and Comparison Data

In [5]:
# Save reference and comparison datasets

reference_path = MONITORING_DIR / "reference_data.csv"
comparison_path = MONITORING_DIR / "comparison_data.csv"
comparison_target_path = MONITORING_DIR / "comparison_target.csv"

X_train.to_csv(reference_path, index=False)
X_test.to_csv(comparison_path, index=False)

y_test.to_frame(name="Churn").to_csv(
    comparison_target_path,
    index=False
)

print("Monitoring datasets saved successfully.")

print("\nSaved files:")
print("-", reference_path.name)
print("-", comparison_path.name)
print("-", comparison_target_path.name)

print("\nReference Data:", X_train.shape)
print("Comparison Data:", X_test.shape)
print("Comparison Target:", y_test.shape)

Monitoring datasets saved successfully.

Saved files:
- reference_data.csv
- comparison_data.csv
- comparison_target.csv

Reference Data: (5634, 19)
Comparison Data: (1409, 19)
Comparison Target: (1409,)


## Identify Feature Types for Monitoring

This section identifies the numerical and categorical features in the reference dataset. Correct feature classification is important because different statistical methods are required to detect distribution changes across different data types.
Binary variables such as SeniorCitizen will be treated as categorical features during monitoring.

In [6]:
# Identify numerical and categorical features for monitoring

numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    column for column in X_train.columns
    if column not in numerical_features
]

# Display feature classification

print("Numerical Features:")
for feature in numerical_features:
    print("-", feature)

print("\nCategorical Features:")
for feature in categorical_features:
    print("-", feature)

print("\nFeature Counts:")
print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))
print("Total:", len(numerical_features) + len(categorical_features))

Numerical Features:
- tenure
- MonthlyCharges
- TotalCharges

Categorical Features:
- gender
- SeniorCitizen
- Partner
- Dependents
- PhoneService
- MultipleLines
- InternetService
- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies
- Contract
- PaperlessBilling
- PaymentMethod

Feature Counts:
Numerical: 3
Categorical: 16
Total: 19


## Statistical Methods for Drift Detection

This section defines the statistical methods used to detect changes in feature distributions.

The Kolmogorov–Smirnov (KS) test will be applied to numerical features, while the Chi-Square test of homogeneity will be used for categorical features.

To control the risk of false positive findings across multiple feature tests, the Bonferroni correction will be applied at a significance level of 0.05.

The analysis will also report effect sizes to help distinguish statistically significant changes from changes that may have limited practical importance.

### Define Drift Detection Functions

In [7]:
from scipy.stats import ks_2samp, chi2_contingency

# Define significance level
ALPHA = 0.05

# Number of feature tests
N_TESTS = len(X_train.columns)

# Bonferroni-adjusted significance level
ADJUSTED_ALPHA = ALPHA / N_TESTS


def calculate_cramers_v(contingency_table, chi2_stat):
    """Calculate Cramer's V for categorical features."""

    n = contingency_table.to_numpy().sum()
    rows, columns = contingency_table.shape

    denominator = n * min(rows - 1, columns - 1)

    if denominator <= 0:
        return 0.0

    return np.sqrt(chi2_stat / denominator)


def detect_numerical_drift(reference, current):
    """Apply the two-sample KS test."""

    result = ks_2samp(
        reference.dropna(),
        current.dropna()
    )

    return {
        "statistic": result.statistic,
        "p_value": result.pvalue
    }


def detect_categorical_drift(reference, current):
    """Apply the Chi-Square test of homogeneity."""

    categories = sorted(
        set(reference.dropna().unique())
        | set(current.dropna().unique()),
        key=str
    )

    contingency_table = pd.DataFrame({
        "Reference": [
            (reference == category).sum()
            for category in categories
        ],
        "Current": [
            (current == category).sum()
            for category in categories
        ]
    }, index=categories).T

    chi2_stat, p_value, _, _ = chi2_contingency(
        contingency_table
    )

    cramers_v = calculate_cramers_v(
        contingency_table,
        chi2_stat
    )

    return {
        "statistic": chi2_stat,
        "p_value": p_value,
        "effect_size": cramers_v
    }


print("Drift detection methods defined successfully.")
print("Significance Level:", ALPHA)
print("Number of Feature Tests:", N_TESTS)
print("Bonferroni-Adjusted Alpha:", round(ADJUSTED_ALPHA, 5))

Drift detection methods defined successfully.
Significance Level: 0.05
Number of Feature Tests: 19
Bonferroni-Adjusted Alpha: 0.00263


### Initial Drift Detection

This section compares the reference data with the test data to identify differences in feature distributions. Numerical and categorical features are evaluated using the statistical methods defined in the previous section.

Since both datasets come from the same original dataset and were randomly split, this comparison is used as an initial baseline check rather than evidence of real production drift.

In [8]:
drift_results = []

for feature in numerical_features:
    result = detect_numerical_drift(
        X_train[feature],
        X_test[feature]
    )

    adjusted_p = min(result["p_value"] * N_TESTS, 1.0)

    drift_results.append({
        "Feature": feature,
        "Feature Type": "Numerical",
        "Test Statistic": result["statistic"],
        "P-Value": result["p_value"],
        "Adjusted P-Value": adjusted_p,
        "Effect Size": result["statistic"],
        "Drift Detected": adjusted_p < ALPHA
    })


for feature in categorical_features:
    result = detect_categorical_drift(
        X_train[feature],
        X_test[feature]
    )

    adjusted_p = min(result["p_value"] * N_TESTS, 1.0)

    drift_results.append({
        "Feature": feature,
        "Feature Type": "Categorical",
        "Test Statistic": result["statistic"],
        "P-Value": result["p_value"],
        "Adjusted P-Value": adjusted_p,
        "Effect Size": result["effect_size"],
        "Drift Detected": adjusted_p < ALPHA
    })


drift_report = pd.DataFrame(drift_results)

drift_report = drift_report.sort_values(
    by="Adjusted P-Value"
).reset_index(drop=True)

display(drift_report.round(5))

print("\nDrift Detection Summary")
print("----------------------")
print("Total Features:", len(drift_report))
print("Features with Drift:", drift_report["Drift Detected"].sum())
print("Features without Drift:", (~drift_report["Drift Detected"]).sum())

,Feature,Feature Type,Test Statistic,P-Value,Adjusted P-Value,Effect Size,Drift Detected
0,tenure,Numerical,0.02118,0.68350,1.0,0.02118,False
1,MonthlyCharges,Numerical,0.03209,0.19115,1.0,0.03209,False
2,TotalCharges,Numerical,0.02958,0.27095,1.0,0.02958,False
3,gender,Categorical,0.37639,0.53954,1.0,0.00731,False
4,SeniorCitizen,Categorical,0.23236,0.62978,1.0,0.00574,False
5,Partner,Categorical,0.17876,0.67244,1.0,0.00504,False
6,Dependents,Categorical,0.29693,0.58581,1.0,0.00649,False
7,PhoneService,Categorical,1.69832,0.19251,1.0,0.01553,False
8,MultipleLines,Categorical,3.35524,0.18682,1.0,0.02183,False
9,InternetService,Categorical,0.26689,0.87508,1.0,0.00616,False



Drift Detection Summary
----------------------
Total Features: 19
Features with Drift: 0
Features without Drift: 19


### Controlled Drift Simulation

This section introduces controlled changes to a copy of the test dataset to evaluate whether the drift detection methods can identify changes in feature distributions.

The simulation modifies selected numerical and categorical features while keeping the original reference and comparison datasets unchanged.

In [9]:
# Create a separate copy for drift simulation
simulated_current_data = X_test.copy()

# Initialize a reproducible random generator
rng = np.random.default_rng(42)

# 1. Introduce numerical drift
simulated_current_data["MonthlyCharges"] *= 1.15

# 2. Introduce categorical drift
sample_size = int(0.30 * len(simulated_current_data))

selected_indices = rng.choice(
    simulated_current_data.index,
    size=sample_size,
    replace=False
)

simulated_current_data.loc[
    selected_indices, "Contract"
] = "Month-to-month"

# Display the changes
print("Controlled drift simulation completed.")

print("\nMonthlyCharges:")
print("Original Mean:", round(X_test["MonthlyCharges"].mean(), 2))
print("Simulated Mean:", round(simulated_current_data["MonthlyCharges"].mean(), 2))

print("\nContract Distribution:")
print("Original:")
display(X_test["Contract"].value_counts(normalize=True).round(3))

print("Simulated:")
display(simulated_current_data["Contract"].value_counts(normalize=True).round(3))

Controlled drift simulation completed.

MonthlyCharges:
Original Mean: 64.09
Simulated Mean: 73.7

Contract Distribution:
Original:


Contract
Month-to-month    0.549
Two year          0.238
One year          0.213
Name: proportion, dtype: float64

Simulated:


Contract
Month-to-month    0.686
Two year          0.163
One year          0.152
Name: proportion, dtype: float64

### Evaluation of Drift Detection Results

This section evaluates the ability of the proposed drift detection mechanism to identify changes in feature distributions.

The statistical tests are applied to the simulated current data and compared against the reference dataset. The Bonferroni correction is used to control the family-wise error rate across all feature tests.

The results are examined using adjusted p-values and effect sizes to determine whether the introduced changes can be detected.

The objective is to verify that the monitoring mechanism can identify controlled distribution changes before applying it to future production data.

In [11]:
simulated_drift_results = []

for feature in numerical_features:
    result = detect_numerical_drift(
        X_train[feature],
        simulated_current_data[feature]
    )

    adjusted_p = min(result["p_value"] * N_TESTS, 1.0)

    simulated_drift_results.append({
        "Feature": feature,
        "Feature Type": "Numerical",
        "Test Statistic": result["statistic"],
        "P-Value": result["p_value"],
        "Adjusted P-Value": adjusted_p,
        "Effect Size": result["statistic"],
        "Drift Detected": adjusted_p < ALPHA
    })


for feature in categorical_features:
    result = detect_categorical_drift(
        X_train[feature],
        simulated_current_data[feature]
    )

    adjusted_p = min(result["p_value"] * N_TESTS, 1.0)

    simulated_drift_results.append({
        "Feature": feature,
        "Feature Type": "Categorical",
        "Test Statistic": result["statistic"],
        "P-Value": result["p_value"],
        "Adjusted P-Value": adjusted_p,
        "Effect Size": result["effect_size"],
        "Drift Detected": adjusted_p < ALPHA
    })


simulated_drift_report = pd.DataFrame(simulated_drift_results)

simulated_drift_report = simulated_drift_report.sort_values(
    by="Adjusted P-Value"
).reset_index(drop=True)

display(simulated_drift_report.round(5))

print("\nControlled Drift Detection Summary")
print("----------------------------------")
print("Total Features:", len(simulated_drift_report))
print(
    "Features with Drift:",
    simulated_drift_report["Drift Detected"].sum()
)
print(
    "Features without Drift:",
    (~simulated_drift_report["Drift Detected"]).sum()
)

,Feature,Feature Type,Test Statistic,P-Value,Adjusted P-Value,Effect Size,Drift Detected
0,MonthlyCharges,Numerical,0.16525,0.00000,0.0,0.16525,True
1,Contract,Categorical,84.67373,0.00000,0.0,0.10965,True
2,TotalCharges,Numerical,0.02958,0.27095,1.0,0.02958,False
3,tenure,Numerical,0.02118,0.68350,1.0,0.02118,False
4,SeniorCitizen,Categorical,0.23236,0.62978,1.0,0.00574,False
5,Partner,Categorical,0.17876,0.67244,1.0,0.00504,False
6,Dependents,Categorical,0.29693,0.58581,1.0,0.00649,False
7,gender,Categorical,0.37639,0.53954,1.0,0.00731,False
8,MultipleLines,Categorical,3.35524,0.18682,1.0,0.02183,False
9,InternetService,Categorical,0.26689,0.87508,1.0,0.00616,False



Controlled Drift Detection Summary
----------------------------------
Total Features: 19
Features with Drift: 2
Features without Drift: 17


In [12]:
# Display the complete simulated drift detection report

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(
    simulated_drift_report.style.format({
        "Test Statistic": "{:.5f}",
        "P-Value": "{:.8f}",
        "Adjusted P-Value": "{:.8f}",
        "Effect Size": "{:.5f}"
    })
)

,Feature,Feature Type,Test Statistic,P-Value,Adjusted P-Value,Effect Size,Drift Detected
0,MonthlyCharges,Numerical,0.16525,0.00000000,0.00000000,0.16525,True
1,Contract,Categorical,84.67373,0.00000000,0.00000000,0.10965,True
2,TotalCharges,Numerical,0.02958,0.27095260,1.00000000,0.02958,False
3,tenure,Numerical,0.02118,0.68349955,1.00000000,0.02118,False
4,SeniorCitizen,Categorical,0.23236,0.62978239,1.00000000,0.00574,False
5,Partner,Categorical,0.17876,0.67243895,1.00000000,0.00504,False
6,Dependents,Categorical,0.29693,0.58581231,1.00000000,0.00649,False
7,gender,Categorical,0.37639,0.53953924,1.00000000,0.00731,False
8,MultipleLines,Categorical,3.35524,0.18681797,1.00000000,0.02183,False
9,InternetService,Categorical,0.26689,0.87507622,1.00000000,0.00616,False


### Drift Detection Robustness Testing

This section evaluates the robustness of the drift detection mechanism under different simulated conditions.

The evaluation includes numerical and categorical changes at different magnitudes (5%, 10%, and 20%). It also generates multiple samples without intentional drift to estimate the false positive rate.

The Bonferroni correction is applied consistently across all 19 feature tests. The results are used to assess the detection capability and reliability of the proposed monitoring mechanism.

In [13]:
# ==========================================
# 1. Configuration
# ==========================================

rng = np.random.default_rng(42)

DRIFT_MAGNITUDES = [0.05, 0.10, 0.20]
NULL_REPETITIONS = 200

# Features selected for controlled testing
numeric_test_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_test_features = {
    "Contract": "Month-to-month",
    "PaymentMethod": "Electronic check"
}

robustness_results = []


# ==========================================
# 2. Helper Function
# ==========================================

def evaluate_all_features(reference, current):

    results = []

    for feature in numerical_features:

        result = detect_numerical_drift(
            reference[feature],
            current[feature]
        )

        p_value = result["p_value"]
        adjusted_p = min(p_value * N_TESTS, 1.0)

        results.append({
            "Feature": feature,
            "Adjusted P-Value": adjusted_p,
            "Drift Detected": adjusted_p < ALPHA
        })

    for feature in categorical_features:

        result = detect_categorical_drift(
            reference[feature],
            current[feature]
        )

        p_value = result["p_value"]
        adjusted_p = min(p_value * N_TESTS, 1.0)

        results.append({
            "Feature": feature,
            "Adjusted P-Value": adjusted_p,
            "Drift Detected": adjusted_p < ALPHA
        })

    return pd.DataFrame(results)


# ==========================================
# 3. Test Different Drift Magnitudes
# ==========================================

for magnitude in DRIFT_MAGNITUDES:

    # Numerical features
    for feature in numeric_test_features:

        current_data = X_test.copy()

        current_data[feature] = (
            current_data[feature] * (1 + magnitude)
        )

        report = evaluate_all_features(
            X_train,
            current_data
        )

        detected = report.loc[
            report["Feature"] == feature,
            "Drift Detected"
        ].iloc[0]

        robustness_results.append({
            "Feature": feature,
            "Feature Type": "Numerical",
            "Drift Magnitude": f"{magnitude:.0%}",
            "Drift Detected": detected
        })

    # Categorical features
    for feature, target_category in categorical_test_features.items():

        current_data = X_test.copy()

        eligible_indices = current_data.index[
            current_data[feature] != target_category
        ]

        number_to_change = int(
            len(current_data) * magnitude
        )

        selected_indices = rng.choice(
            eligible_indices.to_numpy(),
            size=number_to_change,
            replace=False
        )

        current_data.loc[
            selected_indices, feature
        ] = target_category

        report = evaluate_all_features(
            X_train,
            current_data
        )

        detected = report.loc[
            report["Feature"] == feature,
            "Drift Detected"
        ].iloc[0]

        robustness_results.append({
            "Feature": feature,
            "Feature Type": "Categorical",
            "Drift Magnitude": f"{magnitude:.0%}",
            "Drift Detected": detected
        })


robustness_report = pd.DataFrame(robustness_results)


# ==========================================
# 4. Estimate False Positive Rate
# ==========================================

false_positive_counts = pd.Series(
    0,
    index=X_train.columns,
    dtype=int
)

familywise_false_positives = 0

for i in range(NULL_REPETITIONS):

    # Sample from the reference distribution without
    # introducing any intentional distribution change.
    null_sample = X_train.sample(
        n=len(X_test),
        replace=True,
        random_state=int(rng.integers(0, 1_000_000_000))
    )

    null_report = evaluate_all_features(
        X_train,
        null_sample
    )

    detected_features = null_report.loc[
        null_report["Drift Detected"],
        "Feature"
    ].tolist()

    for feature in detected_features:
        false_positive_counts[feature] += 1

    if len(detected_features) > 0:
        familywise_false_positives += 1


false_positive_report = pd.DataFrame({
    "Feature": false_positive_counts.index,
    "False Positive Count": false_positive_counts.values,
    "False Positive Rate": (
        false_positive_counts.values / NULL_REPETITIONS
    )
})

familywise_false_positive_rate = (
    familywise_false_positives / NULL_REPETITIONS
)


# ==========================================
# 5. Summary and Results
# ==========================================

print("DRIFT MAGNITUDE TEST RESULTS")
print("============================")

display(robustness_report)

print("\nDETECTION RATE BY MAGNITUDE")
print("===========================")

magnitude_summary = (
    robustness_report.groupby("Drift Magnitude")
    ["Drift Detected"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

magnitude_summary["Detection Rate"] = (
    magnitude_summary["mean"] * 100
)

display(magnitude_summary.drop(columns="mean").round(2))


print("\nFALSE POSITIVE ANALYSIS")
print("=======================")

display(
    false_positive_report.sort_values(
        "False Positive Rate",
        ascending=False
    ).round(4)
)

print(
    "\nFamily-wise False Positive Rate:",
    f"{familywise_false_positive_rate:.2%}"
)

print("Null Simulations:", NULL_REPETITIONS)
print("Feature Tests per Simulation:", N_TESTS)
print("Adjusted Significance Level:", ADJUSTED_ALPHA)

DRIFT MAGNITUDE TEST RESULTS


,Feature,Feature Type,Drift Magnitude,Drift Detected
0,tenure,Numerical,5%,True
1,MonthlyCharges,Numerical,5%,True
2,TotalCharges,Numerical,5%,False
3,Contract,Categorical,5%,False
4,PaymentMethod,Categorical,5%,False
5,tenure,Numerical,10%,True
6,MonthlyCharges,Numerical,10%,True
7,TotalCharges,Numerical,10%,False
8,Contract,Categorical,10%,True
9,PaymentMethod,Categorical,10%,True



DETECTION RATE BY MAGNITUDE


,Drift Magnitude,sum,count,Detection Rate
0,10%,4,5,80.0
1,20%,5,5,100.0
2,5%,2,5,40.0



FALSE POSITIVE ANALYSIS


,Feature,False Positive Count,False Positive Rate
17,MonthlyCharges,2,0.010
6,MultipleLines,1,0.005
2,Partner,1,0.005
7,InternetService,1,0.005
5,PhoneService,1,0.005
9,OnlineBackup,1,0.005
0,gender,0,0.000
1,SeniorCitizen,0,0.000
3,Dependents,0,0.000
8,OnlineSecurity,0,0.000



Family-wise False Positive Rate: 2.50%
Null Simulations: 200
Feature Tests per Simulation: 19
Adjusted Significance Level: 0.002631578947368421


### Drift Detection Evaluation - Results

The controlled drift simulation confirmed that the proposed mechanism successfully detected changes in MonthlyCharges and Contract, while the other 17 features were not flagged. Further robustness testing showed that the detection rate increased with drift magnitude, reaching 40% at 5%, 80% at 10%, and 100% at 20%.

In 200 simulations without intentional drift, the family-wise false positive rate was 2.50%, below the predefined 5% significance level.

Overall, the results support the use of the proposed statistical methods as an initial drift detection mechanism within ModelGuard. However, further validation using real production data and additional drift scenarios is required to establish its reliability in operational environments.

## Model Performance Monitoring

This section evaluates the predictive performance of the trained XGBoost model using the held-out test dataset.

Several classification metrics are calculated to assess the model's ability to identify customer churn, including Precision, Recall, F1-score, ROC-AUC, and PR-AUC.

The evaluation provides a performance baseline that can be used for comparison when new labeled data becomes available.

In [14]:
# Load the trained XGBoost model
final_xgb_model = joblib.load(
    ARTIFACTS_DIR / "final_xgboost_model.joblib"
)

# Generate prediction probabilities
y_prob = final_xgb_model.predict_proba(X_test)[:, 1]

# Apply the selected classification threshold
SELECTED_THRESHOLD = 0.56
y_pred = (y_prob >= SELECTED_THRESHOLD).astype(int)

# Calculate performance metrics
performance_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-Score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC": average_precision_score(y_test, y_prob)
}

# Display performance results
performance_report = pd.DataFrame(
    performance_metrics.items(),
    columns=["Metric", "Value"]
)

display(performance_report.round(4))

# Display confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

confusion_report = pd.DataFrame(
    cm,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
)

print("Confusion Matrix:")
display(confusion_report)

,Metric,Value
0,Accuracy,0.7644
1,Precision,0.5407
2,Recall,0.7460
3,F1-Score,0.6270
4,ROC-AUC,0.8463
5,PR-AUC,0.6622


Confusion Matrix:


,Predicted No Churn,Predicted Churn
Actual No Churn,798,237
Actual Churn,95,279


### Model Performance Results

The trained XGBoost model achieved an Accuracy of 76.44%, a Recall of 74.60%, and an F1-score of 62.70% on the held-out test dataset. The ROC-AUC and PR-AUC scores were 84.63% and 66.22%, respectively.

The confusion matrix showed that the model correctly identified 279 churn cases, while missing 95 actual churn cases.

These results establish the initial performance baseline for ModelGuard. Future performance monitoring can compare new evaluation results against this baseline when actual customer outcomes become available.

### Save the initial model performance baseline

In [15]:
performance_baseline = {
    "model": "XGBoost",
    "evaluation_dataset": "Held-out Test Dataset",
    "threshold": SELECTED_THRESHOLD,
    "metrics": performance_metrics,
    "confusion_matrix": cm.tolist()
}

baseline_path = MONITORING_DIR / "performance_baseline.json"

with open(baseline_path, "w") as file:
    json.dump(performance_baseline, file, indent=4)

print("Performance baseline saved successfully.")
print("File:", baseline_path)

Performance baseline saved successfully.
File: C:\Users\User\ModelGuard\monitoring\performance_baseline.json


### Batch-Based Model Performance Monitoring

This section evaluates model performance across two separate batches of test data. The results are compared with the initial performance baseline to examine performance consistency across batches.

The experiment uses the actual labels available in the held-out test dataset. Since both batches are randomly sampled from the same dataset, this represents a simulated monitoring scenario rather than real production monitoring.

In [16]:
# ==========================================
# 1. Split Test Data into Two Batches
# ==========================================

from sklearn.model_selection import train_test_split

X_batch1, X_batch2, y_batch1, y_batch2 = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

batches = {
    "Batch 1": (X_batch1, y_batch1),
    "Batch 2": (X_batch2, y_batch2)
}


# ==========================================
# 2. Evaluate Model Performance
# ==========================================

batch_results = []

for batch_name, (X_batch, y_batch) in batches.items():

    probabilities = final_xgb_model.predict_proba(X_batch)[:, 1]

    predictions = (
        probabilities >= SELECTED_THRESHOLD
    ).astype(int)

    batch_results.append({
        "Batch": batch_name,
        "Samples": len(y_batch),
        "Accuracy": accuracy_score(y_batch, predictions),
        "Precision": precision_score(
            y_batch, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_batch, predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            y_batch, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_batch, probabilities),
        "PR-AUC": average_precision_score(
            y_batch, probabilities
        )
    })


# ==========================================
# 3. Compare Batch Results with Baseline
# ==========================================

baseline_row = {
    "Batch": "Initial Baseline",
    "Samples": len(y_test),
    **performance_metrics
}

comparison_report = pd.DataFrame(
    [baseline_row] + batch_results
)

display(comparison_report.round(4))


# ==========================================
# 4. Calculate Performance Differences
# ==========================================

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC",
    "PR-AUC"
]

batch_comparison = comparison_report.set_index("Batch")

difference_report = pd.DataFrame({
    "Batch 1 Difference": (
        batch_comparison.loc["Batch 1", metric_columns]
        - batch_comparison.loc["Initial Baseline", metric_columns]
    ),
    "Batch 2 Difference": (
        batch_comparison.loc["Batch 2", metric_columns]
        - batch_comparison.loc["Initial Baseline", metric_columns]
    )
})

print("Performance Differences Compared with Baseline:")
display(difference_report.round(4))


# ==========================================
# 5. Save Monitoring Results
# ==========================================

batch_report_path = MONITORING_DIR / "batch_performance_report.csv"

comparison_report.to_csv(batch_report_path, index=False)

print("Batch performance monitoring completed.")
print("Results saved to:", batch_report_path)

,Batch,Samples,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,Initial Baseline,1409,0.7644,0.5407,0.7460,0.6270,0.8463,0.6622
1,Batch 1,704,0.7500,0.5204,0.7487,0.6140,0.8499,0.6802
2,Batch 2,705,0.7787,0.5628,0.7433,0.6406,0.8422,0.6452


Performance Differences Compared with Baseline:


,Batch 1 Difference,Batch 2 Difference
Accuracy,-0.0144,0.0144
Precision,-0.0203,0.0221
Recall,0.0027,-0.0027
F1-Score,-0.0129,0.0136
ROC-AUC,0.0036,-0.0040
PR-AUC,0.0180,-0.0171


Batch performance monitoring completed.
Results saved to: C:\Users\User\ModelGuard\monitoring\batch_performance_report.csv


### Batch Performance Monitoring Results

The batch-based evaluation showed relatively consistent model performance across the two test batches. Recall remained stable at 74.87% and 74.33%, while the F1-score was 61.40% for Batch 1 and 64.06% for Batch 2, compared with 62.70% for the initial baseline.

No substantial performance deterioration was observed in this experiment. These results demonstrate how ModelGuard can compare model performance across separate data batches. However, since both batches were randomly sampled from the same test dataset, the experiment does not represent actual performance changes over time.